# 12 - Fast animated dashboard

Run notebook 11 first, then run these cells in order.

In [ ]:
html_code = '''<style>.title-box{font-family:Arial,sans-serif;font-size:42px;font-weight:bold;color:#fff;text-align:center;padding:25px;border-radius:18px;background:linear-gradient(120deg,#06152d,#124b90,#06152d);background-size:200% 100%;animation:flow 3s linear infinite,glow 1.4s ease-in-out infinite alternate}@keyframes flow{0%{background-position:0% 50%}100%{background-position:200% 50%}}@keyframes glow{from{box-shadow:0 0 10px #4cc9ff}to{box-shadow:0 0 30px #7be8ff}}</style><div class=title-box>⚡ Smart Meter Dashboard</div>'''
class HtmlView:
 def __init__(self,content): self.content=content
 def _repr_html_(self): return self.content
 def __repr__(self): return 'HTML renderer unavailable.'
HtmlView(html_code)

## Bronze KPIs

In [ ]:
import os,html
CATALOG=os.getenv('AIDP_CATALOG','aidp_poc')
m=spark.table(f'{CATALOG}.gold.dashboard_metrics').first()
b=spark.table(f'{CATALOG}.gold.dashboard_bronze_categories').orderBy('events',ascending=False).collect()
rows=''.join(f'<tr><td>{html.escape(str(x.meter_type))}</td><td>{html.escape(str(x.customer_segment))}</td><td>{x.events:,}</td><td>{x.active_meters:,}</td></tr>' for x in b)
page=f'''<style>.panel{{font-family:Arial;color:#fff;background:#071c3d;padding:20px;border-radius:16px}}.grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}}.card{{background:#0d2854;border:1px solid #5693db;border-radius:12px;padding:16px}}.label{{font-size:11px;color:#d5e7ff}}.value{{font-size:28px;font-weight:bold;color:#fff;margin-top:7px}}table{{width:100%;margin-top:16px;border-collapse:collapse}}td,th{{padding:9px;text-align:left;border-bottom:1px solid #ffffff33}}td:nth-child(n+3),th:nth-child(n+3){{text-align:right}}</style><div class=panel><h2>01 · Bronze event landing</h2><div class=grid><div class=card><div class=label>RAW EVENTS RECEIVED</div><div class=value>{m.bronze_events:,}</div></div><div class=card><div class=label>ACTIVE METERS IN BRONZE</div><div class=value>{m.bronze_active_meters:,}</div></div><div class=card><div class=label>LATEST INGEST</div><div class=value>{html.escape(str(m.latest_ingested))}</div></div></div><h3>Events and active meters by category</h3><table><tr><th>Meter type</th><th>Customer category</th><th>Events</th><th>Active meters</th></tr>{rows}</table></div>'''
HtmlView(page)

## Silver and Gold KPIs

In [ ]:
s=spark.table(f'{CATALOG}.gold.dashboard_segments').orderBy('kwh',ascending=False).collect()
rows=''.join(f'<tr><td>{html.escape(str(x.customer_segment))}</td><td>{x.meters:,}</td><td>{x.kwh:,.2f}</td><td>{x.peak_kw:,.2f}</td></tr>' for x in s)
page=f'''<style>.panel{{font-family:Arial;color:#fff;background:#071c3d;padding:20px;border-radius:16px;margin-top:10px}}.grid{{display:grid;grid-template-columns:repeat(4,1fr);gap:14px}}.card{{background:#0d2854;border:1px solid #5693db;border-radius:12px;padding:16px}}.label{{font-size:11px;color:#d5e7ff}}.value{{font-size:28px;font-weight:bold;color:#fff;margin-top:7px}}table{{width:100%;margin-top:16px;border-collapse:collapse}}td,th{{padding:9px;text-align:left;border-bottom:1px solid #ffffff33}}td:nth-child(n+2),th:nth-child(n+2){{text-align:right}}</style><div class=panel><h2>02 · Silver validation and Gold energy</h2><div class=grid><div class=card><div class=label>VALID READINGS</div><div class=value>{m.silver_readings:,}</div></div><div class=card><div class=label>ACTIVE METERS</div><div class=value>{m.active_meters:,}</div></div><div class=card><div class=label>15-MIN INTERVALS</div><div class=value>{m.intervals:,}</div></div><div class=card><div class=label>PEAK DEMAND</div><div class=value>{m.peak_kw or 0} kW</div></div></div><h3>Latest interval energy by segment</h3><table><tr><th>Segment</th><th>Meters</th><th>Energy kWh</th><th>Peak kW</th></tr>{rows}</table></div>'''
HtmlView(page)

## ML forecast KPIs

In [ ]:
p=spark.table(f'{CATALOG}.gold.dashboard_predictions').limit(100).collect()
rows=''.join(f'<tr><td>{html.escape(str(x.meter_id))}</td><td>{x.prediction_kwh:,.4f}</td><td>{html.escape(str(x.interval_start_utc))}</td></tr>' for x in p)
page=f'''<style>.panel{{font-family:Arial;color:#fff;background:#071c3d;padding:20px;border-radius:16px;margin-top:10px}}.grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}}.card{{background:#0d2854;border:1px solid #5693db;border-radius:12px;padding:16px}}.label{{font-size:11px;color:#d5e7ff}}.value{{font-size:28px;font-weight:bold;color:#fff;margin-top:7px}}table{{width:100%;margin-top:16px;border-collapse:collapse}}td,th{{padding:8px;text-align:left;border-bottom:1px solid #ffffff33}}td:nth-child(n+2),th:nth-child(n+2){{text-align:right}}</style><div class=panel><h2>03 · ML next-interval forecast</h2><div class=grid><div class=card><div class=label>MODEL VERSION</div><div class=value>{html.escape(str(m.model_version))}</div></div><div class=card><div class=label>PREDICTIONS</div><div class=value>{m.prediction_count:,}</div></div><div class=card><div class=label>AVERAGE FORECAST</div><div class=value>{m.avg_prediction_kwh or 0} kWh</div></div></div><h3>Next 15-minute forecasts · first 100 meters</h3><table><tr><th>Meter</th><th>Forecast kWh</th><th>Future interval</th></tr>{rows}</table></div>'''
HtmlView(page)